In [1]:
import requests
from requests.auth import HTTPBasicAuth
import pandas as pd
from calendar import monthrange
import time
import datetime
import zipfile
import io
import os

In [2]:
import urllib3
from urllib3.exceptions import InsecureRequestWarning

urllib3.disable_warnings(InsecureRequestWarning)

In [3]:
caiso_tac_subba_map = {
    "SDGE-TAC": "SDGE",
    "PGE-TAC": "PGAE",
    "VEA-TAC": "VEA",
    "SCE-TAC": "SCE",
    "MWD-TAC": "SCE"
}

In [4]:
def send_query(dt_start, market_run_id):
    dt_end = dt_start + pd.Timedelta(days=29)

    start_time = dt_start.strftime("%Y%m%dT%H:%M")
    end_time = dt_end.strftime("%Y%m%dT%H:%M")
    api_url = f"http://oasis.caiso.com/oasisapi/SingleZip?queryname=SLD_FCST&market_run_id={market_run_id}&startdatetime={start_time}-0000&enddatetime={end_time}-0000&version=1&resultformat=6"
    response = requests.get(api_url, verify=False)
    return response, dt_end

def get_caiso_profile_for_year(year, market_run_id):
    df_list = []
    dt_start = datetime.datetime(year, 1, 1) - pd.Timedelta(hours=1)
    while dt_start.year <= year:
        response, dt_start = send_query(dt_start, market_run_id)
        time.sleep(4)
        
        assert response.status_code == 200, response.status_code
        zfile = zipfile.ZipFile(io.BytesIO(response.content))
        with (
            zfile.open(
                response.headers['Content-Disposition']
                .split('filename=')
                [1]
                .replace(';', '')
                .replace('.zip', '.csv')
            )
            as csv_fname
        ):
            df = pd.read_csv(csv_fname)
            df = (
                df.loc[df.TAC_AREA_NAME.str.contains("TAC")]
                .sort_values("INTERVALSTARTTIME_GMT")
                [['INTERVALSTARTTIME_GMT', 'INTERVALENDTIME_GMT', 'TAC_AREA_NAME', 'MW']]
            )
        
        df_list.append(df)
    
    df = pd.concat(df_list, ignore_index=True)
    df['subba'] = df['TAC_AREA_NAME'].map(caiso_tac_subba_map)
    df['timestamp'] = (
        pd.to_datetime(df['INTERVALENDTIME_GMT'])
        .dt
        .tz_localize(None)
    )
    df = (
        df.loc[df.timestamp.dt.year == year]
        .rename(columns={'MW': 'value'})
        .groupby(['timestamp', 'subba'])
        .sum(numeric_only=True)
        .reset_index()
    )
    
    return df

In [5]:
caiso_subba_load_profiles_list = []
for year in range(2015, 2025):
    df = get_caiso_profile_for_year(year, market_run_id='ACTUAL')
    caiso_subba_load_profiles_list.append(df)

In [6]:
caiso_subba_forecast_profiles_list = []
for year in range(2015, 2025):
    df = get_caiso_profile_for_year(year, market_run_id='DAM')
    caiso_subba_forecast_profiles_list.append(df)

In [ ]:
caiso_load = pd.concat(caiso_subba_load_profiles_list, ignore_index=True)
caiso_load.to_csv(f"../data/iso_load_profiles/caiso.csv", index=False)

caiso_forecast = pd.concat(caiso_subba_forecast_profiles_list, ignore_index=True)
caiso_forecast.to_csv(f"../data/iso_load_profiles/caiso_forecast.csv", index=False)